# A10b Preliminary vs Final — Score Comparison (CPU)

Membandingkan metrik **preliminary (locked silver test)** dengan **final
(human-gold test)** untuk ketiga model deteksi aspek (Keyword, TF-IDF,
IndoBERT). Memakai split leakage-safe yang sama; yang berubah hanya label
referensi (silver -> gold).

Membaca metrik silver (`*-silver-v1-test-metrics.json`) dan gold
(`*-gold-v1-test-metrics.json`) lalu menghasilkan tabel + figure perbandingan.

Prasyarat: notebook `05` (silver) dan `10` (gold) sudah menghasilkan metriknya di
Drive (`SIPATURE/metrics/`). IndoBERT-on-gold masih pending (butuh GPU).


## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter

In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"

PROJECT_DIR = Path("/content/hackathon/ml")
METRICS_DIR = PROJECT_DIR / "artifacts" / "metrics"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "comparison"

DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "comparison"

METRIC_FILES = [
    "keyword-silver-v1-test-metrics.json",
    "tfidf-silver-v1-test-metrics.json",
    "keyword-gold-v1-test-metrics.json",
    "tfidf-gold-v1-test-metrics.json",
]

print("Drive root:", DRIVE_ROOT)
print("Sumber metrics:", DRIVE_METRICS_DIR)
print("Figure dir (lokal):", FIGURE_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber metrics: /content/drive/MyDrive/SIPATURE/metrics
Figure dir (lokal): /content/hackathon/ml/artifacts/figures/comparison


## Step 3 — Clone repository dari GitHub

In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)

In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
50d2f82 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
edcfa4b fix: validate gold.jsonl without annotator_id (freeze-gold output has none)
cf4ee2d feat: preliminary vs final score comparison (module + CLI + notebook 11)


## Step 5 — Install dependencies

In [5]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 6 — Verifikasi versi package

In [3]:
import matplotlib

print("Matplotlib:", matplotlib.__version__)


Matplotlib: 3.10.3


## Step 7 — Copy metrics (silver + gold) dari Drive ke lokal

In [4]:
import shutil
from pathlib import Path

METRICS_DIR.mkdir(parents=True, exist_ok=True)
for filename in METRIC_FILES:
    source = DRIVE_METRICS_DIR / filename
    assert source.is_file(), f"Metric file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, METRICS_DIR / filename)
    print("Disalin:", filename)

# IndoBERT gold (opsional — hanya ada bila notebook 12 sudah dijalankan).
indobert_gold = DRIVE_METRICS_DIR / "indobert-gold-v1-test-metrics.json"
if indobert_gold.is_file():
    shutil.copy2(indobert_gold, METRICS_DIR / indobert_gold.name)
    print("Disalin:", indobert_gold.name)


Disalin: keyword-silver-v1-test-metrics.json
Disalin: tfidf-silver-v1-test-metrics.json
Disalin: keyword-gold-v1-test-metrics.json
Disalin: tfidf-gold-v1-test-metrics.json


## Step 8 — Import modul sipature_ml

In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Jalankan perbandingan preliminary vs final

In [6]:
from sipature_ml.comparison import run_preliminary_final_comparison

summary = run_preliminary_final_comparison(METRICS_DIR, FIGURE_DIR)

for model, stages in summary["models"].items():
    pre = stages["preliminary"]
    fin = stages["final"]
    pre_txt = f"macro {pre['macro_f1']:.4f}" if pre else "n/a"
    fin_txt = f"macro {fin['macro_f1']:.4f}" if fin else "pending"
    print(f"{model:<18} preliminary={pre_txt:<20} final={fin_txt}")


Keyword            preliminary=macro 0.9768         final=macro 0.7056
TF-IDF             preliminary=macro 0.7201         final=macro 0.5777
IndoBERT (aspek)   preliminary=macro 0.5247         final=pending


## Step 10 — Tampilkan tabel & delta

In [7]:
print(f"{'model':<18}{'silver':>10}{'gold':>10}{'delta':>10}")
print("-" * 48)
for model, stages in summary["models"].items():
    pre = stages["preliminary"]
    fin = stages["final"]
    if pre is None or fin is None:
        print(f"{model:<18}{'n/a':>10}{'pending':>10}{'-':>10}")
        continue
    delta = fin["macro_f1"] - pre["macro_f1"]
    print(f"{model:<18}{pre['macro_f1']:>10.4f}{fin['macro_f1']:>10.4f}{delta:>+10.4f}")

print("\nNotes:")
for note in summary["notes"]:
    print(" -", note)
print("\nFigure:", summary["figure"])


model                 silver      gold     delta
------------------------------------------------
Keyword               0.9768    0.7056   -0.2712
TF-IDF                0.7201    0.5777   -0.1424
IndoBERT (aspek)         n/a   pending         -

Notes:
 - Keyword silver Macro F1 0.9768 is circular against silver rules.
 - IndoBERT polarity (silver 0.7459) is a separate task and not in the aspect comparison.
 - IndoBERT final (gold) is pending GPU/Colab execution.

Figure: 37_preliminary_vs_final_macro_f1.png


## Step 11 — Copy output ke Drive

In [8]:
import shutil
from pathlib import Path

DRIVE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for source in sorted(FIGURE_DIR.glob("*")):
    if source.is_file():
        shutil.copy2(source, DRIVE_FIGURE_DIR / source.name)
        print(f"Disalin: {source.name} -> {DRIVE_FIGURE_DIR}")


Disalin: 37_preliminary_vs_final_macro_f1.png -> /content/drive/MyDrive/SIPATURE/figures/comparison
Disalin: preliminary_final_comparison.json -> /content/drive/MyDrive/SIPATURE/figures/comparison


## Step 12 — Run summary (hash & metric)

In [9]:
# ============================================================
# RUN SUMMARY — path output dan temuan kunci.
# ============================================================
from pathlib import Path
from sipature_ml.manifest import sha256_file

summary_path = FIGURE_DIR / "preliminary_final_comparison.json"
print("SUMMARY JSON  :", summary_path)
print("SUMMARY SHA256 :", sha256_file(summary_path))
print("FIGURE        :", FIGURE_DIR / summary["figure"])

print("\nTEMUAN KUNCI:")
print(" - Keyword silver 0.9768 bersifat circular; di gold turun ke 0.7056.")
print(" - TF-IDF turun 0.7201 -> 0.5777 (delta -0.1424).")
print(" - Di gold, keyword > TF-IDF (0.71 vs 0.58) — kebalikan dari silver.")
print(" - IndoBERT-on-gold pending (GPU).")


SUMMARY JSON  : /content/hackathon/ml/artifacts/figures/comparison/preliminary_final_comparison.json
SUMMARY SHA256 : 7abbd5ba9c3b43a35856c6a20976de9e8ac54480ac644e3eabc58ee88d948513
FIGURE        : /content/hackathon/ml/artifacts/figures/comparison/37_preliminary_vs_final_macro_f1.png

TEMUAN KUNCI:
 - Keyword silver 0.9768 bersifat circular; di gold turun ke 0.7056.
 - TF-IDF turun 0.7201 -> 0.5777 (delta -0.1424).
 - Di gold, keyword > TF-IDF (0.71 vs 0.58) — kebalikan dari silver.
 - IndoBERT-on-gold pending (GPU).
